In [3]:
# Menghubungkan Google Colab dengan Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Memanggil library yang dibutuhkan
import zipfile
import io
import re
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Lokasi file ZIP di Google Drive
ZIP_PATH = '/content/drive/MyDrive/archive (4).zip'

# Agar hasil pembagian data bisa diulang dengan hasil yang sama
RANDOM_STATE = 42

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
# Nama file Excel di dalam ZIP
file_dataset = {
    'CNN': 'Cleaned/dataset_cnn_10k_cleaned.xlsx',
    'Kompas': 'Cleaned/dataset_kompas_4k_cleaned.xlsx',
    'Tempo': 'Cleaned/dataset_tempo_6k_cleaned.xlsx',
    'TurnBackHoax': 'Cleaned/dataset_turnbackhoax_10_cleaned.xlsx'
}

# Tempat untuk menampung masing-masing dataset
list_data = []

# Membuka ZIP dan membaca setiap file Excel
with zipfile.ZipFile(ZIP_PATH, 'r') as zip_file:
    for nama_sumber, nama_file in file_dataset.items():
        file_excel = zip_file.open(nama_file)
        data_sumber = pd.read_excel(file_excel)

        # Menambahkan nama sumber agar asal setiap baris dapat diperiksa
        data_sumber['source'] = nama_sumber
        list_data.append(data_sumber)

# Menggabungkan empat dataset menjadi satu tabel
data = pd.concat(list_data, ignore_index=True)

print('Jumlah baris dan kolom:', data.shape)
display(data.head())

Jumlah baris dan kolom: (31353, 13)


,Unnamed: 0,Title,Timestamp,FullText,Tags,Author,Url,text_new,hoax,source,politik,Narasi,Clean Narasi
0,0,Anies di Milad BKMT: Pengajian Menghasilkan Ib...,"Selasa, 21 Feb 2023 21:22 WIB","Jakarta, CNN Indonesia -- Mantan Gubernur DKI ...",anies baswedan;pengajian;pilpres 2024;badan ko...,CNN Indonesia,https://www.cnnindonesia.com/nasional/20230221...,Anies di Milad BKMT: Pengajian Menghasilkan Ib...,0,CNN,NaN,NaN,NaN
1,1,Edy Soal Pilgub Sumut: Kalau yang Maju Abal-ab...,"Selasa, 21 Feb 2023 20:46 WIB","Medan, CNN Indonesia -- Gubernur Sumatera Utar...",edy rahmayadi;pemilu 2024;pilkada 2024,CNN Indonesia,https://www.cnnindonesia.com/nasional/20230221...,Edy Soal Pilgub Sumut: Kalau yang Maju Abal-ab...,0,CNN,NaN,NaN,NaN
2,2,PKB Bakal Daftarkan Menaker Ida Fauziyah Jadi ...,"Selasa, 21 Feb 2023 20:33 WIB","Jakarta, CNN Indonesia -- Partai Kebangkitan B...",ida fauziyah;pkb;pemilu 2024;pileg 2024,CNN Indonesia,https://www.cnnindonesia.com/nasional/20230221...,PKB Bakal Daftarkan Menaker Ida Fauziyah Jadi ...,0,CNN,NaN,NaN,NaN
3,3,Gede Pasek Doakan AHY Jadi Capres atau Cawapres,"Selasa, 21 Feb 2023 19:58 WIB","Jakarta, CNN Indonesia -- Ketua Umum Partai Ke...",gede pasek suardika;ahy;pilpres 2024;pemilu 20...,CNN Indonesia,https://www.cnnindonesia.com/nasional/20230221...,Gede Pasek Doakan AHY Jadi Capres atau Cawapre...,0,CNN,NaN,NaN,NaN
4,4,PKN Siapkan Jabatan Khusus Buat Anas Urbaningr...,"Selasa, 21 Feb 2023 18:56 WIB","Jakarta, CNN Indonesia -- Dewan Pimpinan Pusat...",anas urbaningrum;pkn;pemilu 2024,CNN Indonesia,https://www.cnnindonesia.com/nasional/20230221...,PKN Siapkan Jabatan Khusus Buat Anas Urbaningr...,0,CNN,NaN,NaN,NaN


In [5]:
#@title Pemeriksaan awal dataset
print('Jumlah data awal:', len(data))

print('\nInformasi kolom dan tipe data:')
data.info()

print('\nJumlah nilai kosong setiap kolom:')
display(data.isna().sum().to_frame('Missing Values'))

print('\nJumlah data berdasarkan sumber dan label:')
display(pd.crosstab(data['source'], data['hoax'], margins=True))

Jumlah data awal: 31353

Informasi kolom dan tipe data:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31353 entries, 0 to 31352
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Unnamed: 0    31353 non-null  int64  
 1   Title         31332 non-null  object 
 2   Timestamp     31332 non-null  object 
 3   FullText      31326 non-null  object 
 4   Tags          31191 non-null  object 
 5   Author        31016 non-null  object 
 6   Url           31353 non-null  object 
 7   text_new      20945 non-null  object 
 8   hoax          31353 non-null  int64  
 9   source        31353 non-null  object 
 10  politik       10381 non-null  float64
 11  Narasi        10381 non-null  object 
 12  Clean Narasi  6502 non-null   object 
dtypes: float64(1), int64(2), object(10)
memory usage: 3.1+ MB

Jumlah nilai kosong setiap kolom:


,Missing Values
Unnamed: 0,0
Title,21
Timestamp,21
FullText,27
Tags,162
Author,337
Url,0
text_new,10408
hoax,0
source,0



Jumlah data berdasarkan sumber dan label:


hoax,0,1,All
source,,,
CNN,9630,0,9630
Kompas,4750,0,4750
Tempo,6592,0,6592
TurnBackHoax,0,10381,10381
All,20972,10381,31353


In [6]:
judul_turnbackhoax = data.loc[data['source'] == 'TurnBackHoax', 'Title']
penanda = judul_turnbackhoax.str.extract(r'^\s*\[([^\]]+)\]', expand=False)
display(penanda.value_counts().head(10).to_frame('Jumlah'))


,Jumlah
Title,
SALAH,8025
BENAR,425
HOAX,394
KLARIFIKASI,318
DISINFORMASI,184
BERITA,175
FALSE,129
EDUKASI,42
Top 5,34


In [7]:
#@title Data Cleaning
# Mengubah Title dan FullText menjadi tipe teks
for kolom in ['Title', 'FullText']:
    data[kolom] = data[kolom].astype('string').str.strip()

    # Mengubah teks kosong menjadi missing value
    data.loc[data[kolom] == '', kolom] = pd.NA

# Memastikan kolom label berupa angka
data['hoax'] = pd.to_numeric(data['hoax'], errors='coerce')

# Menghitung jumlah baris sebelum dihapus
jumlah_awal = len(data)

# Menghapus baris yang kosong pada kolom wajib
data = data.dropna(subset=['Title', 'FullText', 'hoax']).copy()

print('Baris yang dihapus:', jumlah_awal - len(data))

# Mempertahankan hanya label 0 dan 1
data = data[data['hoax'].isin([0, 1])].copy()
data['hoax'] = data['hoax'].astype(int)

print('Jumlah data setelah menangani missing values:', len(data))
print('Tipe data kolom utama:')
print(data[['Title', 'FullText', 'hoax']].dtypes)

Baris yang dihapus: 27
Jumlah data setelah menangani missing values: 31326
Tipe data kolom utama:
Title       string[python]
FullText    string[python]
hoax                 int64
dtype: object


In [8]:
#@title Menghapus data duplikat

# Membuat kunci pemeriksaan dari judul dan isi berita.
# Huruf dibuat kecil dan spasi berulang dirapikan agar perbedaan penulisan kecil tidak dihitung sebagai data berbeda.
data['judul_key'] = data['Title'].str.lower().str.replace(r'\s+', ' ', regex=True).str.strip()
data['isi_key'] = data['FullText'].str.lower().str.replace(r'\s+', ' ', regex=True).str.strip()

jumlah_sebelum = len(data)

# Menghapus berita yang memiliki judul dan isi sama
data = data.drop_duplicates(subset=['judul_key', 'isi_key']).copy()

print('Jumlah duplikat yang dihapus:', jumlah_sebelum - len(data))

# Kolom bantuan tidak diperlukan lagi
data = data.drop(columns=['judul_key', 'isi_key'])
data = data.reset_index(drop=True)

print('Jumlah data setelah menghapus duplikat:', len(data))

Jumlah duplikat yang dihapus: 5
Jumlah data setelah menghapus duplikat: 31321


In [9]:
#@title Memeriksa outliers pada panjang teks

# Menghitung jumlah kata pada isi berita
data['jumlah_kata_isi'] = data['FullText'].str.split().str.len()

# Menghitung batas outlier menggunakan metode IQR
Q1 = data['jumlah_kata_isi'].quantile(0.25)
Q3 = data['jumlah_kata_isi'].quantile(0.75)
IQR = Q3 - Q1

batas_bawah = Q1 - (1.5 * IQR)
batas_atas = Q3 + (1.5 * IQR)

# Menandai data yang berada di luar batas
outlier = (data['jumlah_kata_isi'] < batas_bawah) | (data['jumlah_kata_isi'] > batas_atas)

print('Batas bawah:', batas_bawah)
print('Batas atas:', batas_atas)
print('Jumlah data terindikasi outlier:', outlier.sum())

# Outlier tidak dihapus otomatis karena artikel panjang masih bisa valid.

Batas bawah: 13.5
Batas atas: 633.5
Jumlah data terindikasi outlier: 1756


In [10]:
#@title hasil Data Cleaning
print('Jumlah data setelah cleaning:', len(data))
print('\nSisa missing values pada kolom wajib:')
display(data[['Title', 'FullText', 'hoax']].isna().sum().to_frame('Missing Values'))
print('\nDistribusi label setelah cleaning:')
display(data['hoax'].value_counts().sort_index().rename(index={0: 'Valid (0)', 1: 'Hoaks (1)'}).to_frame('Jumlah Data'))

Jumlah data setelah cleaning: 31321

Sisa missing values pada kolom wajib:


,Missing Values
Title,0
FullText,0
hoax,0



Distribusi label setelah cleaning:


,Jumlah Data
hoax,
Valid (0),20945
Hoaks (1),10376


In [11]:
#@title Feature Engineering
# Menghapus penanda di awal judul, misalnya [SALAH], [BENAR], atau [HOAX].
# Penanda seperti ini berisiko memberi petunjuk langsung mengenai label.
data['judul_tanpa_penanda'] = data['Title'].str.replace(
    r'^\s*\[[^\]]+\]\s*', '', regex=True
).str.strip()

# Menggabungkan judul dan isi menjadi satu kolom teks
data['text_raw'] = data['judul_tanpa_penanda'] + ' ' + data['FullText']

# Membuat fitur tambahan untuk mengetahui panjang teks
data['jumlah_karakter'] = data['text_raw'].str.len()
data['jumlah_kata'] = data['text_raw'].str.split().str.len()

# Menentukan input (X) dan target/label (y)
X = data['text_raw']
y = data['hoax']

display(data[['Title', 'judul_tanpa_penanda', 'text_raw', 'jumlah_kata', 'hoax']].head(3))

,Title,judul_tanpa_penanda,text_raw,jumlah_kata,hoax
0,Anies di Milad BKMT: Pengajian Menghasilkan Ib...,Anies di Milad BKMT: Pengajian Menghasilkan Ib...,Anies di Milad BKMT: Pengajian Menghasilkan Ib...,300,0
1,Edy Soal Pilgub Sumut: Kalau yang Maju Abal-ab...,Edy Soal Pilgub Sumut: Kalau yang Maju Abal-ab...,Edy Soal Pilgub Sumut: Kalau yang Maju Abal-ab...,332,0
2,PKB Bakal Daftarkan Menaker Ida Fauziyah Jadi ...,PKB Bakal Daftarkan Menaker Ida Fauziyah Jadi ...,PKB Bakal Daftarkan Menaker Ida Fauziyah Jadi ...,222,0


In [12]:
#@title Data Transformation
# Fungsi untuk membersihkan teks berita

def clean_text(teks):
    teks = str(teks).lower()                         # 1. Mengubah huruf menjadi kecil
    teks = re.sub(r'https?://\S+|www\.\S+', ' ', teks)  # 2. Menghapus URL
    teks = re.sub(r'<[^>]+>', ' ', teks)             # 3. Menghapus tag HTML
    teks = re.sub(r'[^a-z0-9\s]', ' ', teks)         # 4. Menghapus tanda baca/simbol
    teks = re.sub(r'\s+', ' ', teks).strip()         # 5. Merapikan spasi
    return teks

# Menerapkan fungsi ke seluruh berita
data['text_clean'] = data['text_raw'].apply(clean_text)

# Menghapus baris yang menjadi kosong setelah dibersihkan
data = data[data['text_clean'].str.strip() != ''].copy()
data = data.reset_index(drop=True)

# Memperbarui X dan y setelah transformasi
X = data['text_clean'].astype(str)
y = data['hoax'].astype(int)

display(data[['text_raw', 'text_clean', 'hoax']].head(3))

,text_raw,text_clean,hoax
0,Anies di Milad BKMT: Pengajian Menghasilkan Ib...,anies di milad bkmt pengajian menghasilkan ibu...,0
1,Edy Soal Pilgub Sumut: Kalau yang Maju Abal-ab...,edy soal pilgub sumut kalau yang maju abal aba...,0
2,PKB Bakal Daftarkan Menaker Ida Fauziyah Jadi ...,pkb bakal daftarkan menaker ida fauziyah jadi ...,0


In [13]:
#@title Membagi data sebelum membentuk TF-IDF dan tokenisasi
# Membagi data menjadi 70% training dan 30% data sementara
X_train, X_sisa, y_train, y_sisa = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

# Membagi 30% data sementara menjadi 15% validation dan 15% test
X_val, X_test, y_val, y_test = train_test_split(
    X_sisa, y_sisa,
    test_size=0.50,
    random_state=42,
    stratify=y_sisa
)

print('Jumlah training   :', len(X_train))
print('Jumlah validation :', len(X_val))
print('Jumlah test       :', len(X_test))

Jumlah training   : 21924
Jumlah validation : 4698
Jumlah test       : 4699


In [14]:
#@title TF-IDF untuk Naive Bayes dan SVM
# Membuat pengubah teks menjadi bobot TF-IDF
tfidf = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2)
)

# Belajar kosakata dan bobot hanya dari data training
X_train_tfidf = tfidf.fit_transform(X_train)

# Validation dan test hanya diubah memakai hasil dari training
X_val_tfidf = tfidf.transform(X_val)
X_test_tfidf = tfidf.transform(X_test)

print('Bentuk data TF-IDF training:', X_train_tfidf.shape)
print('Bentuk data TF-IDF validation:', X_val_tfidf.shape)
print('Bentuk data TF-IDF test:', X_test_tfidf.shape)

Bentuk data TF-IDF training: (21924, 20000)
Bentuk data TF-IDF validation: (4698, 20000)
Bentuk data TF-IDF test: (4699, 20000)


In [15]:
#@title Tokenisasi dan padding untuk LSTM
# Batas jumlah kata dalam kamus dan panjang urutan
MAX_WORDS = 20000
MAX_LEN = 300

# Membuat tokenizer dan menyediakan penanda untuk kata yang tidak dikenal
tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token='<OOV>')

# Mempelajari kamus kata hanya dari data training
tokenizer.fit_on_texts(X_train.tolist())

# Mengubah setiap teks menjadi urutan angka
X_train_seq = tokenizer.texts_to_sequences(X_train.tolist())
X_val_seq = tokenizer.texts_to_sequences(X_val.tolist())
X_test_seq = tokenizer.texts_to_sequences(X_test.tolist())

# Menyamakan panjang urutan menjadi 300 angka
X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding='post', truncating='post')
X_val_pad = pad_sequences(X_val_seq, maxlen=MAX_LEN, padding='post', truncating='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=MAX_LEN, padding='post', truncating='post')

print('Bentuk data LSTM training:', X_train_pad.shape)
print('Bentuk data LSTM validation:', X_val_pad.shape)
print('Bentuk data LSTM test:', X_test_pad.shape)

Bentuk data LSTM training: (21924, 300)
Bentuk data LSTM validation: (4698, 300)
Bentuk data LSTM test: (4699, 300)


In [16]:
#@title Data Balancing
# Melihat jumlah masing-masing kelas pada data training
jumlah_kelas = pd.Series(y_train).value_counts().sort_index()
display(jumlah_kelas.rename(index={0: 'Valid (0)', 1: 'Hoaks (1)'}).to_frame('Jumlah Data Training'))

# Menghitung bobot kelas berdasarkan data training saja
kelas = np.unique(y_train)
bobot = compute_class_weight(
    class_weight='balanced',
    classes=kelas,
    y=y_train
)

# Membuat dictionary agar bobot mudah digunakan saat model dilatih
class_weight_dict = {int(k): float(v) for k, v in zip(kelas, bobot)}

print('Bobot setiap kelas:', class_weight_dict)

,Jumlah Data Training
hoax,
Valid (0),14661
Hoaks (1),7263


Bobot setiap kelas: {0: 0.7476979742173112, 1: 1.5092936802973977}
